# About this notebook

This notebook is created by Bella Ratmelia (bellar@smu.edu.sg) for SMU Libraries' Python 101: Tinkering with Pandas' DataFrame Skill Myself Up workshop.

## Dataset

Dataset used in this workshop is a snippet of World Values Survey wave 7. You can download the data prepared for this workshop from [**this URL**](https://raw.githubusercontent.com/bellaratmelia/introductory-r-socsci/refs/heads/main/data/wvs-wave7-sg-ca-nz.csv), and save it in your local PC.

***Note: I've edited the data file to have missing values and other imperfections for the purpose of this workshop***

In [ ]:
# import the necessary packages
import pandas as pd

We can run this shortcut below to get Google Colab to download the data and save it to our Colab environment.

**If you run this notebook locally (in Visual Studio or Jupyter Notebook), download the CSV from the link above, and you can skip the cell below.**

`!wget` is a linux command, and we indicate as such to the jupyter notebook by prepending it with exclamation mark.

In [ ]:
!wget "https://raw.githubusercontent.com/bellaratmelia/smulib-python101/refs/heads/main/2026-09%20Python101-DataFrame/wvs-wave7-sg-ca-nz.csv"

# Handling Tabular Data

In [ ]:
# load/read CSV to Python via DataFrame
# make sure you've run the !wget cell above first, so the file is in your Colab session
# (on your own PC instead? download the CSV from the link above and put it next to this notebook)
data = pd.read_csv('wvs-wave7-sg-ca-nz.csv')

In [ ]:
data

## Preliminary checks

In [ ]:
# find out more about a dataframe.
data.info()

In [ ]:
# Get the summary statistics of the columns that have numerical data.
# All other columns are ignored, unless you use the argument include='all'.

data.describe()

In [ ]:
# How many missing values are in each column?
# (recall: this file was intentionally given some gaps for the workshop)
data.isna().sum()

## Renaming columns

In [ ]:
# Check all the columns name

data.columns

In [ ]:
# Sometimes column names need to be renamed to make it easier for us
# e.g. when a column name is too long or doesn't make sense,
# we can rename them to something more meaningful

data = data.rename(columns={
    "country": "country_code"
})

data.columns


## Selecting specific columns of dataframe

In [ ]:
# Selecting a subset ("slicing")
# get the age of participants
# two ways to do the same thing:
data["age"]     # bracket notation; always works!
data.age        # dot notation is convenient, but breaks on spaces/reserved names

In [ ]:
# Describe just a column
data.age.describe()

In [ ]:
# get the marital_status and age of participants
data[["marital_status", "age"]]

## Filtering the rows to fit specified criteria

In [ ]:
# Filtering: Get all data from participants who thinks work is important
criteria = data["work_importance"] >= 2
data_work_impt = data[criteria]

data_work_impt

In [ ]:
# Get participants who thinks work is important AND is male
criteria = (data["work_importance"] >= 2) & (data["sex"] == "Male")
data_work_male = data[criteria]

data_work_male

In [ ]:
# query method
data_work_male = data.query("work_importance >= 2 and sex == 'Male'")
data_work_male

**FYI — `.query()` vs the `&` / `|` style:** both give the *same result*, so which you use is a trade-off, not a right/wrong choice.

- 👍 **Pros:** far more readable, especially with several conditions; you can write `and` / `or` in plain English, and there's no need to wrap each condition in `( )`.
- 👎 **Cons:** the condition is just a text string, so typos hide *inside* it instead of erroring clearly, and it's harder to build dynamically (e.g. when a column name is stored in a variable).

There's rarely one "correct" method in pandas, but each method comes with trade-off. Knowing *why* you'd pick one is the part that's yours to bring (an AI will happily generate either without telling you which fits).

## Filtering for both rows and columns

In [ ]:
# Even more granular filtering with .loc, we can filter rows and columns criteria at one go

criteria = (data["country_code"] == "SGP")
sgp_life_sats = data.loc[criteria, "life_satisfaction"]

sgp_life_sats

### 📊 Slido checkpoint → Q1–Q4
*First half — four quick questions:*
1. *Preliminary checks — what `.info()` / `.describe()` / the missing-value count tell you.*
2. *Selecting columns — Series vs DataFrame (`[ ]` vs `[[ ]]`), and dot vs bracket.*
3. *Filtering — the `and` vs `&` error, and an AI-written `|`/`&` bug that runs but lies.*
4. *`.loc` with a mask — selecting rows and a column together, all by label.*

## Handling empty values

In [ ]:
# get participants whose religiousity is known
criteria = data["religiousity"].notna()
religiousity_known = data[criteria]

religiousity_known

In [ ]:
# we can also update the values in dataframe, especially for the empty ones

data["religiousity"] = data["religiousity"].fillna("Unknown")
data["religiousity"].describe()

## Counting and Sorting

In [ ]:
# Find out distribution of religiousity of participants
data["religiousity"].value_counts()

In [ ]:
# Grouped by religiousity and marital status
data[["religiousity", "marital_status"]].value_counts()

In [ ]:
# sort the data by age, highest first.
# We reassign to a new variable instead of using inplace=True — inplace is being
# phased out in pandas, and it would permanently reorder the shared `data`.
data_by_age = data.sort_values(by="age", ascending=False)
data_by_age.head(15)

## Creating and dropping columns

In [ ]:
# create a new column
data["employment_marital"] = data["employment"] + "-" + data["marital_status"]

data["employment_marital"].head(10)

In [ ]:
# drop a column we no longer need
data = data.drop(columns=["employment_marital"])

data.columns

## Average, median, mode

In [ ]:
data["life_satisfaction"].mean()

In [ ]:
data["life_satisfaction"].median()

In [ ]:
data["marital_status"].mode()

In [ ]:
# difference between sexes when it comes to life_satisfaction
grouped_data = data.groupby(by=["sex"])
grouped_data['life_satisfaction'].mean()

## Saving to CSV + simple visualizations

In [ ]:
# saving the dataframe in its current state to a CSV
# index=False keeps pandas from writing the row numbers as an extra column
data.to_csv("wvs-edited.csv", index=False)

In [ ]:
data['country_code'].value_counts().plot(kind='bar', xlabel='Country', ylabel='Count')

### 📊 Slido checkpoint → Q5–Q8
*Second half — four quick questions:*
1. *Missing values — drop vs fill as an analytical choice, not a syntax one.*
2. *Counting & grouping — "the average for **each** country" points at `groupby`.*
3. *Creating & dropping columns — adding a derived column, then `drop(columns=...)`.*
4. *Saving & quick viz — `index=False` when writing CSV, and `value_counts()` → bar chart.*

# Handling Time Series Data

## Download and load the data

For this section, we will be using a time series data about monthly unemployment rate in Singapore since 1948 to 2024 for 4 different age groups. The original data is retrieved from Singstats. You can download a modified CSV from [this URL](https://raw.githubusercontent.com/bellaratmelia/smulib-python101/refs/heads/main/2026-09%20Python101-DataFrame/unemployment-age.csv)

In [ ]:
# Linux command to download the data to the environment.

!wget "https://raw.githubusercontent.com/bellaratmelia/smulib-python101/refs/heads/main/2026-09%20Python101-DataFrame/unemployment-age.csv"

In [ ]:
unemployment_data = pd.read_csv('unemployment-age.csv')

In [ ]:
# check the loaded data
unemployment_data.info()

## Set the date column as index

In [ ]:
# set the date as a datetime object
unemployment_data['date'] = pd.to_datetime(unemployment_data['date'])
unemployment_data.info()

In [ ]:
# set the index
unemployment_data.set_index('date', inplace=True)
unemployment_data.info()

In [ ]:
# check the data and the latest date
unemployment_data.tail(10)

In [ ]:
# Drop the last column. It's a stray empty column with no proper name,
# so pandas loaded it as "Unnamed: 5" — we drop it by that name.
unemployment_data = unemployment_data.drop(columns="Unnamed: 5")

# confirm removal
print(unemployment_data.info())

## Manipulating data

In [ ]:
# retrieving data on a specific year / month / date
unemployment_data.loc['2015-10-01']

In [ ]:
unemployment_data.loc['2015']

In [ ]:
unemployment_data.loc['2013-01':'2013-06']

In [ ]:
# get all the 2013 data for 25 - 54 years age group
unemployment_data.loc['2013', "25_54yrs"]

In [ ]:
unemployment_data.loc['2013', ["20_24yrs","25_54yrs"]]

## Visualize Time Series

In [ ]:
import matplotlib.pyplot as plt
# matplotlib package for visualization

In [ ]:
unemployment_data.loc['2013':'2015', ["20_24yrs","25_54yrs"]].plot()
plt.show()

## Exercises to try yourself!

1. Get the marital status of singaporeans who rated life satisfaction higher than 6
2. Sort participants based on country code in descending order
3. Plot a bar chart that visualizes the data based on country and sexes

In [ ]:
# answer to qn 1
criteria = (data["life_satisfaction"] > 6) & (data["country_code"] == "SGP")
country_data = data.loc[criteria, "marital_status"]

country_data

In [ ]:
# answer to qn 2
sorted_by_country = data.sort_values(by="country_code", ascending=False)
sorted_by_country.head(15)

In [ ]:
# answer to qn 3
data[['country_code', 'sex']].value_counts().plot(kind='bar', xlabel='Country, Sex', ylabel='Count')